In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ds-605-first-development-challenge/sample_submission - sample_submission.csv.csv
/kaggle/input/competitions/ds-605-first-development-challenge/train.csv
/kaggle/input/competitions/ds-605-first-development-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ds-605-first-development-challenge/sample_submission - sample_submission.csv.csv
/kaggle/input/competitions/ds-605-first-development-challenge/train.csv
/kaggle/input/competitions/ds-605-first-development-challenge/test.csv


In [4]:
import pandas as pd

train_path = "/kaggle/input/competitions/ds-605-first-development-challenge/train.csv"
test_path = "/kaggle/input/competitions/ds-605-first-development-challenge/test.csv"

original_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training data shape:", original_df.shape)
print("Test data shape:", test_df.shape)

Training data shape: (9864, 19)
Test data shape: (2466, 18)


In [5]:
original_df.head()

,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


In [6]:
print("Training columns:")
print(original_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

Training columns:
['Session_ID', 'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

Test columns:
['Session_ID', 'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']


In [7]:
original_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Session_ID               9864 non-null   int64  
 1   Administrative           9864 non-null   int64  
 2   Administrative_Duration  9372 non-null   float64
 3   Informational            9864 non-null   int64  
 4   Informational_Duration   9864 non-null   float64
 5   ProductRelated           9864 non-null   int64  
 6   ProductRelated_Duration  9864 non-null   float64
 7   BounceRates              9864 non-null   float64
 8   ExitRates                9173 non-null   float64
 9   PageValues               9864 non-null   float64
 10  SpecialDay               9864 non-null   float64
 11  Month                    9864 non-null   object 
 12  OperatingSystems         9864 non-null   int64  
 13  Browser                  9864 non-null   int64  
 14  Region                  

In [8]:
missing_values = original_df.isnull().sum()

print(missing_values)

Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691
PageValues                   0
SpecialDay                   0
Month                        0
OperatingSystems             0
Browser                      0
Region                     591
TrafficType                789
VisitorType                394
Weekend                      0
Revenue                      0
dtype: int64


In [9]:
print("Total missing values:", original_df.isnull().sum().sum())

Total missing values: 2957


In [10]:
print("Total missing values in test:", test_df.isnull().sum().sum())

Total missing values in test: 740


In [11]:
print(original_df["Revenue"].value_counts())
print("\nPercentage:")
print(original_df["Revenue"].value_counts(normalize=True) * 100)

Revenue
False    8211
True     1653
Name: count, dtype: int64

Percentage:
Revenue
False    83.242092
True     16.757908
Name: proportion, dtype: float64


In [12]:
print("Duplicate rows:", original_df.duplicated().sum())

Duplicate rows: 0


In [13]:
X = original_df.drop(columns=["Revenue", "Session_ID"])

y = original_df["Revenue"].astype(int)

X_test = test_df.drop(columns=["Session_ID"])

print(y.value_counts())

Revenue
0    8211
1    1653
Name: count, dtype: int64


In [14]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Categorical features:
['Month', 'VisitorType', 'Weekend']


In [17]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numeric pipeline created successfully!")

Numeric pipeline created successfully!


In [18]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical pipeline created successfully!")

Categorical pipeline created successfully!


In [19]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessor created successfully!")

Preprocessor created successfully!


In [20]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))

Training rows: 7891
Validation rows: 1973


In [21]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Logistic Regression model created!")

Logistic Regression model created!


In [22]:
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [23]:
val_predictions = model.predict(X_val)

print("First 20 predictions:")
print(val_predictions[:20])

First 20 predictions:
[0 1 0 0 1 0 0 0 0 1 1 1 0 0 0 0 0 1 0 0]


In [24]:
accuracy = accuracy_score(y_val, val_predictions)

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.8504815002534212


In [25]:
print(classification_report(y_val, val_predictions))

              precision    recall  f1-score   support

           0       0.94      0.88      0.91      1642
           1       0.54      0.70      0.61       331

    accuracy                           0.85      1973
   macro avg       0.74      0.79      0.76      1973
weighted avg       0.87      0.85      0.86      1973



In [26]:
model.fit(X, y)

print("Final model trained successfully!")

Final model trained successfully!


In [27]:
predictions = model.predict(X_test)

print("Number of predictions:", len(predictions))
print("Number of test rows:", len(test_df))

Number of predictions: 2466
Number of test rows: 2466


In [28]:
print("Prediction distribution:")
print(pd.Series(predictions).value_counts())

Prediction distribution:
0    1926
1     540
Name: count, dtype: int64


In [29]:
fitted_preprocessor = model.named_steps["preprocessor"]

X_processed = fitted_preprocessor.transform(X)

if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

processed_df = pd.DataFrame(X_processed)

print("Processed data shape:", processed_df.shape)

Processed data shape: (9864, 29)


In [30]:
processed_missing = processed_df.isnull().sum().sum()

print("Missing values after preprocessing:", processed_missing)

Missing values after preprocessing: 0


In [31]:
original_rows = original_df.shape[0]
processed_rows = processed_df.shape[0]

row_retained_percent = (
    processed_rows / original_rows
) * 100

print("Original rows:", original_rows)
print("Processed rows:", processed_rows)
print("Rows retained:", round(row_retained_percent, 2), "%")

Original rows: 9864
Processed rows: 9864
Rows retained: 100.0 %


In [32]:
print("original_df:", original_df.shape)
print("processed_df:", processed_df.shape)
print("model:", type(model).__name__)
print("predictions:", len(predictions))
print("test_df:", test_df.shape)

original_df: (9864, 19)
processed_df: (9864, 29)
model: Pipeline
predictions: 2466
test_df: (2466, 18)


In [33]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Checkpoint information
# ---------------------------------------------------

original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (
    processed_rows / original_rows
) * 100


# ---------------------------------------------------
# Detect model used
# ---------------------------------------------------

final_model = model

# Handle GridSearchCV / RandomizedSearchCV
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

# Handle sklearn Pipeline
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__


# ---------------------------------------------------
# Create checkpoint section
# ---------------------------------------------------

checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})


# ---------------------------------------------------
# Create prediction section
# ---------------------------------------------------

prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})


# ---------------------------------------------------
# Combine and save
# ---------------------------------------------------

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)

submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  29
6  row_retained_percent               100.0
7            model_name  LogisticRegression


In [34]:
submission = pd.read_csv("submission.csv")

print("Submission shape:", submission.shape)
print("\nFirst 15 rows:")
print(submission.head(15))

Submission shape: (2474, 2)

First 15 rows:
                      id               value
0       original_missing                2957
1      processed_missing                   0
2          original_rows                9864
3         processed_rows                9864
4       original_columns                  19
5      processed_columns                  29
6   row_retained_percent               100.0
7             model_name  LogisticRegression
8                 106094                   0
9                 111845                   0
10                106794                   1
11                103444                   0
12                106833                   0
13                102684                   0
14                110590                   0


In [35]:
print("Columns:", submission.columns.tolist())
print("Test rows:", len(test_df))
print("Prediction rows:", len(submission) - 8)
print("Unique test IDs:", test_df["Session_ID"].nunique())
print("Unique submission IDs:", submission["id"].nunique())

Columns: ['id', 'value']
Test rows: 2466
Prediction rows: 2466
Unique test IDs: 2466
Unique submission IDs: 2474


In [36]:
print("Submission file created successfully!")
print("File name: submission.csv")
print("Rows in submission:", len(submission))
print("Columns:", submission.columns.tolist())
print("Model:", model_name)
print("Rows retained:", round(row_retained_percent, 2), "%")
print("Processed missing values:", processed_missing)

Submission file created successfully!
File name: submission.csv
Rows in submission: 2474
Columns: ['id', 'value']
Model: LogisticRegression
Rows retained: 100.0 %
Processed missing values: 0
